In [2]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import lpips
import torch
from torch.utils.data import DataLoader, Dataset
from typing import Dict, Tuple, List, Optional, Union
from model_freetech import RawFormer
import warnings
warnings.filterwarnings("ignore")

# -------------------------- 配置参数（与单个正确代码完全一致） --------------------------
# 基础目录（daylight根目录）
BASE_DIR = "eval_dataset/extreme_dark"
# 分目录配置（RAW和RGB文件分别存放）
RAW_DIR = os.path.join(BASE_DIR, "RAW")       # RAW文件路径：eval_dataset/daylight/RAW
RGB_DIR = os.path.join(BASE_DIR, "RGB")       # BMP文件路径：eval_dataset/daylight/RGB
ENHANCE_DIR = os.path.join(BASE_DIR, "ENHANCE")  # 增强图保存路径：eval_dataset/daylight/ENHANCE

# 输出配置
ENHANCED_PREFIX = "enhanced_"
OUTPUT_EXCEL = os.path.join(BASE_DIR, "enhanced_image_evaluation.xlsx")  # Excel保存到daylight目录

# 模型和设备配置（与单个正确代码一致）
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LPIPS_MODEL = "alex"
MODEL_SIZE = "S"  # 训练时的model_size
MODEL_DIM = 32 if MODEL_SIZE == "S" else 48 if MODEL_SIZE == "B" else 64
CHECKPOINT_PATH = "result/Freetech/weights/model_best.pth"  # 与单个正确代码一致

# 数据配置（与单个正确代码完全一致！）
RAW_WIDTH = 1920
RAW_HEIGHT = 1280
RGB_WIDTH = 1920
RGB_HEIGHT = 1280
RAW_HEADER_SIZE = 0  # 与单个正确代码一致
NORMALIZE_RANGE = (0, 1)  # 与单个正确代码一致

# 增强功能配置
RAW_SUFFIX = ".raw"  # RAW文件后缀
RGB_SUFFIX = ".bmp"  # GT文件后缀（BMP）
SAVE_FORMAT = "bmp"  # 增强图保存格式：仅保留BMP（与单个代码逻辑一致）

# ----------------------------------------------------------------

# 确保输出目录存在
os.makedirs(ENHANCE_DIR, exist_ok=True)
print(f"增强图将保存到：{ENHANCE_DIR}")
print(f"Excel结果将保存到：{OUTPUT_EXCEL}")

class RawRGB_Paired_SeparateDir(Dataset):
    """适配RAW和RGB分目录存储的数据集类（完全复刻单个正确代码的处理逻辑）"""
    def __init__(
        self,
        raw_dir: str,
        rgb_dir: str,
        raw_width: int,
        raw_height: int,
        raw_header_size: int = 0,
        rgb_width: int = 1920,
        rgb_height: int = 1280,
        normalize_range: Tuple[float, float] = (0, 1),
        raw_suffix: str = ".raw",
        rgb_suffix: str = ".bmp"
    ):
        self.raw_dir = raw_dir
        self.rgb_dir = rgb_dir
        self.raw_width = raw_width
        self.raw_height = raw_height
        self.raw_header_size = raw_header_size
        self.rgb_width = rgb_width
        self.rgb_height = rgb_height
        self.min_val, self.max_val = normalize_range
        self.raw_suffix = raw_suffix.lower()
        self.rgb_suffix = rgb_suffix.lower()

        # 扫描RAW目录下的所有RAW文件，并匹配RGB目录下的同名GT文件
        self.raw_rgb_pairs = self._scan_paired_files()
        if len(self.raw_rgb_pairs) == 0:
            raise ValueError(
                f"未找到配对的 {raw_suffix} 和 {rgb_suffix} 文件\n"
                f"检查目录：\n- RAW: {raw_dir}\n- RGB: {rgb_dir}\n"
                f"确保文件名完全一致（仅后缀不同）"
            )
        print(f"找到 {len(self.raw_rgb_pairs)} 组有效配对文件")

    def _scan_paired_files(self) -> List[Tuple[str, str]]:
        """扫描RAW目录，匹配RGB目录下的同名GT文件"""
        paired_files = []
        
        # 遍历RAW目录下的所有RAW文件
        for raw_filename in os.listdir(self.raw_dir):
            if raw_filename.lower().endswith(self.raw_suffix):
                # 提取基础文件名（不含后缀）
                base_name = os.path.splitext(raw_filename)[0]
                # 构建RGB目录下的GT文件路径
                rgb_filename = f"{base_name}{self.rgb_suffix}"
                rgb_path = os.path.join(self.rgb_dir, rgb_filename)
                raw_path = os.path.join(self.raw_dir, raw_filename)
                
                # 检查GT文件是否存在
                if os.path.exists(rgb_path):
                    paired_files.append((raw_path, rgb_path))
                else:
                    print(f"警告：跳过 {raw_filename} - RGB目录中未找到 {rgb_filename}")
        
        return paired_files

    def _load_raw(self, raw_path: str) -> Optional[np.ndarray]:
        """加载RAW文件（完全复刻单个正确代码的逻辑）"""
        try:
            with open(raw_path, 'rb') as f:
                f.seek(self.raw_header_size)
                raw_data = np.fromfile(f, dtype=np.uint16)
            # 确保数据量足够
            expected_size = self.raw_height * self.raw_width
            if len(raw_data) < expected_size:
                print(f"警告：RAW文件 {os.path.basename(raw_path)} 数据不完整（实际：{len(raw_data)}，期望：{expected_size}）")
                return None
            raw_img = raw_data[:expected_size].reshape(self.raw_height, self.raw_width)
            return raw_img.astype(np.float32)
        except Exception as e:
            print(f"错误：加载RAW文件 {os.path.basename(raw_path)} 失败 - {str(e)}")
            return None

    def _normalize_raw(self, raw_img: np.ndarray) -> np.ndarray:
        """RAW归一化（完全复刻单个正确代码的公式）"""
        return raw_img.astype(np.float32) / 255.0
        # 若训练时启用了黑电平处理，请注释上面一行，启用下面的训练公式：
        # ap = 100
        # black_level = 512
        # max_raw = 16383
        # raw_processed = (np.maximum(raw_img.astype(np.float32) - black_level, 0) / (max_raw - black_level)) * ap
        # return raw_processed

    def _load_rgb(self, rgb_path: str) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        """加载GT图（完全复刻单个正确代码的逻辑）"""
        try:
            bmp_img = cv2.imread(rgb_path, cv2.IMREAD_COLOR)
            if bmp_img is None:
                print(f"警告：GT文件 {os.path.basename(rgb_path)} 损坏或无法读取")
                return None, None
            
            bmp_img = cv2.cvtColor(bmp_img, cv2.COLOR_BGR2RGB)
            # 调整尺寸到训练时的大小（与单个正确代码一致）
            if bmp_img.shape[:2] != (self.rgb_height, self.rgb_width):
                bmp_img = cv2.resize(bmp_img, (self.rgb_width, self.rgb_height), interpolation=cv2.INTER_LINEAR)
            
            bmp_uint8 = bmp_img.astype(np.uint8)
            bmp_norm = bmp_uint8.astype(np.float32) / 255.0  # 与单个正确代码一致：(0,1)范围
            return bmp_norm, bmp_uint8
        except Exception as e:
            print(f"错误：加载GT文件 {os.path.basename(rgb_path)} 失败 - {str(e)}")
            return None, None

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, np.ndarray, str]:
        """返回 (input_raw, gt_rgb, gt_uint8, base_name) - 完全复刻单个正确代码的张量格式"""
        raw_path, rgb_path = self.raw_rgb_pairs[idx]
        base_name = os.path.splitext(os.path.basename(raw_path))[0]

        # 加载数据（增加有效性检查）
        raw_img = self._load_raw(raw_path)
        bmp_norm, bmp_uint8 = self._load_rgb(rgb_path)
        if raw_img is None or bmp_norm is None or bmp_uint8 is None:
            raise ValueError(f"样本 {base_name} 数据加载失败")

        # RAW归一化（与单个正确代码一致）
        raw_norm = self._normalize_raw(raw_img)

        # 转换为张量（严格复刻单个正确代码的格式）
        input_raw = torch.from_numpy(raw_norm).unsqueeze(0).float()  # (1, H, W) - 与单个代码一致
        gt_rgb = torch.from_numpy(bmp_norm.transpose(2, 0, 1)).float()  # (3, H, W) - 与单个代码一致

        return input_raw, gt_rgb, bmp_uint8, base_name

    def __len__(self) -> int:
        return len(self.raw_rgb_pairs)

# -------------------------- 模型加载（完全复刻单个正确代码） --------------------------
def load_enhance_model() -> RawFormer:
    num_virtual_cams = 5  # 与单个正确代码一致
    pretrain_mode = False
    model = RawFormer(
        dim=MODEL_DIM,
        use_rep_nr=True,
        num_virtual_cams=num_virtual_cams,
        pretrain_mode=pretrain_mode
    )
    # 加载权重（与单个正确代码一致）
    try:
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
        state_dict = checkpoint["state_dict"]
        new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(new_state_dict, strict=False)
        model = model.to(DEVICE)
        model.eval()
        print(f"模型已加载到 {DEVICE}，权重路径：{CHECKPOINT_PATH}")
        return model
    except Exception as e:
        print(f"错误：加载模型失败 - {str(e)}")
        raise SystemExit(1)

# -------------------------- 数据加载（适配分目录，保持单个代码的处理逻辑） --------------------------
def create_eval_dataset() -> Tuple[DataLoader, List[str]]:
    # 检查RAW和RGB目录是否存在
    if not os.path.exists(RAW_DIR):
        raise FileNotFoundError(f"RAW目录不存在：{RAW_DIR}")
    if not os.path.exists(RGB_DIR):
        raise FileNotFoundError(f"RGB目录不存在：{RGB_DIR}")
    
    dataset = RawRGB_Paired_SeparateDir(
        raw_dir=RAW_DIR,
        rgb_dir=RGB_DIR,
        raw_width=RAW_WIDTH,
        raw_height=RAW_HEIGHT,
        raw_header_size=RAW_HEADER_SIZE,
        rgb_width=RGB_WIDTH,
        rgb_height=RGB_HEIGHT,
        normalize_range=NORMALIZE_RANGE,
        raw_suffix=RAW_SUFFIX,
        rgb_suffix=RGB_SUFFIX
    )
    dataloader = DataLoader(
        dataset,
        batch_size=1,  # 保持batch=1，与单个代码一致
        shuffle=False,
        num_workers=0,  # 单线程，避免数据错乱（与单个代码一致）
        pin_memory=True if torch.cuda.is_available() else False,
        drop_last=False
    )
    base_names = [os.path.splitext(os.path.basename(f[0]))[0] for f in dataset.raw_rgb_pairs]
    return dataloader, base_names

# -------------------------- 图像增强（完全复刻单个代码的推理逻辑，仅增加保存） --------------------------
def enhance_raw_batch(
    input_raw: torch.Tensor, 
    model: RawFormer, 
    base_name: str,
    save_dir: str = ENHANCE_DIR
) -> Tuple[Optional[np.ndarray], Optional[str], Optional[torch.Tensor]]:
    """增强RAW图像（完全复刻单个正确代码的推理逻辑），保存到ENHANCE目录"""
    try:
        # 模型推理（与单个正确代码完全一致）
        with torch.no_grad():
            pred_rgb_tensor = model(input_raw, cam_id=0)  # 保留batch维度：(1,3,H,W)
            pred_rgb_tensor = torch.clamp(pred_rgb_tensor, 0, 1)  # 与单个代码一致
        
        # 转换为uint8（保持维度正确性，避免增强图发黑）
        # 步骤：(1,3,H,W) → 压缩batch维度 → (3,H,W) → 转置为 (H,W,3) → 乘255 → uint8
        pred_rgb_np = pred_rgb_tensor.cpu().numpy()  # (1,3,H,W)
        pred_rgb_np = pred_rgb_np.squeeze(0)  # 仅压缩batch维度 → (3,H,W)（关键：不破坏通道维度）
        pred_rgb_np = pred_rgb_np.transpose((1, 2, 0))  # (H,W,3)
        pred_rgb_np = (pred_rgb_np * 255).astype(np.uint8)  # 与单个代码一致的数值范围转换
        
        # 保存增强图（BMP格式，无损，与单个代码逻辑一致）
        enhanced_path = os.path.join(save_dir, f"{ENHANCED_PREFIX}{base_name}.bmp")
        cv2.imwrite(enhanced_path, cv2.cvtColor(pred_rgb_np, cv2.COLOR_RGB2BGR))
        
        print(f"增强图已保存：{os.path.basename(enhanced_path)}")
        return pred_rgb_np, enhanced_path, pred_rgb_tensor  # 返回原始维度的张量 (1,3,H,W)
    except Exception as e:
        print(f"错误：增强图像 {base_name} 失败 - {str(e)}")
        return None, None, None

# -------------------------- 指标计算（完全复刻单个正确代码的函数） --------------------------
def calculate_psnr_train_style(gt_rgb_tensor: torch.Tensor, pred_rgb_tensor: torch.Tensor) -> float:
    """完全复刻单个正确代码的PSNR计算逻辑"""
    # 验证输入维度（确保与单个代码一致）
    assert gt_rgb_tensor.shape == pred_rgb_tensor.shape, f"维度不匹配：gt={gt_rgb_tensor.shape}, pred={pred_rgb_tensor.shape}"
    assert gt_rgb_tensor.shape[0] == 1, f"batch维度错误：{gt_rgb_tensor.shape[0]}"
    assert gt_rgb_tensor.shape[1] == 3, f"通道维度错误：{gt_rgb_tensor.shape[1]}"
    
    gt_np = gt_rgb_tensor.cpu().numpy().transpose(0, 2, 3, 1)  # (1, H, W, 3)
    pred_np = np.clip(pred_rgb_tensor.detach().cpu().numpy().transpose(0, 2, 3, 1), 0.0, 1.0)
    
    # 转uint8（与单个正确代码一致）
    gt_uint8 = (gt_np * 255).astype(np.uint8)
    pred_uint8 = (pred_np * 255).astype(np.uint8)
    
    psnr_value = psnr(gt_uint8, pred_uint8, data_range=255)
    return psnr_value[0] if isinstance(psnr_value, np.ndarray) else psnr_value

def calculate_ssim(gt_uint8: np.ndarray, pred_uint8: np.ndarray) -> float:
    """完全复刻单个正确代码的SSIM计算"""
    return ssim(
        gt_uint8, pred_uint8,
        data_range=255,
        channel_axis=2,
        gaussian_weights=True,
        sigma=1.5
    )

def calculate_lpips(gt_uint8: np.ndarray, pred_uint8: np.ndarray, lpips_model) -> float:
    """完全复刻单个正确代码的LPIPS计算"""
    def preprocess(img: np.ndarray) -> torch.Tensor:
        img_norm = (img / 255.0) * 2 - 1  # 0-255 → [-1,1]（与单个代码一致）
        return torch.from_numpy(img_norm.transpose((2, 0, 1))).float().unsqueeze(0).to(DEVICE)
    
    gt_tensor = preprocess(gt_uint8)
    pred_tensor = preprocess(pred_uint8)
    
    with torch.no_grad():
        dist = lpips_model(gt_tensor, pred_tensor).item()
    return dist

def calculate_white_balance_error(gt_uint8: np.ndarray, pred_uint8: np.ndarray) -> float:
    """完全复刻单个正确代码的WBE计算"""
    gt = gt_uint8.astype(np.float32) / 255.0
    pred = pred_uint8.astype(np.float32) / 255.0
    
    # 计算RGB通道均值
    gt_r, gt_g, gt_b = np.mean(gt[..., 0]), np.mean(gt[..., 1]), np.mean(gt[..., 2])
    pred_r, pred_g, pred_b = np.mean(pred[..., 0]), np.mean(pred[..., 1]), np.mean(pred[..., 2])
    
    # 避免除以零
    gt_g = max(gt_g, 1e-6)
    pred_g = max(pred_g, 1e-6)
    
    # 计算R/G、B/G比值误差
    return abs((pred_r/pred_g) - (gt_r/gt_g)) + abs((pred_b/pred_g) - (gt_b/gt_g))

# -------------------------- 主流程（完全复刻单个正确代码的逻辑，仅增加批量遍历） --------------------------
def main():
    print("="*60)
    print(f"开始批量评估（完全复刻单个正确代码逻辑）")
    print(f"RAW文件目录：{RAW_DIR}")
    print(f"GT文件目录：{RGB_DIR}")
    print(f"增强图目录：{ENHANCE_DIR}")
    print("="*60)

    # 加载模型（与单个正确代码一致）
    print("\n加载增强模型...")
    enhance_model = load_enhance_model()
    
    # 初始化LPIPS（与单个正确代码一致）
    print(f"\n初始化LPIPS模型（{LPIPS_MODEL}）...")
    try:
        lpips_model = lpips.LPIPS(net=LPIPS_MODEL).to(DEVICE)
        lpips_model.eval()
    except Exception as e:
        print(f"错误：初始化LPIPS失败 - {str(e)}")
        return
    
    # 加载数据集（分目录结构，保持单个代码的处理逻辑）
    print("\n加载评估数据集...")
    try:
        eval_dataloader, base_names = create_eval_dataset()
    except (ValueError, FileNotFoundError) as e:
        print(f"错误：{e}")
        return
    
    # 批量处理（逐个样本处理，完全复刻单个代码的逻辑）
    results = []
    total_samples = len(eval_dataloader)
    print(f"\n开始处理 {total_samples} 个样本...")
    print("="*60)
    
    for idx, (input_raw, gt_rgb, gt_uint8, base_name) in enumerate(eval_dataloader, 1):
        print(f"\n[{idx}/{total_samples}] 处理样本：{base_name}")
        
        try:
            # 数据移到设备（与单个正确代码一致，保持batch维度）
            input_raw = input_raw.to(DEVICE)  # (1,1,H,W)
            gt_rgb = gt_rgb.to(DEVICE)        # (1,3,H,W) → 与单个代码一致
            gt_uint8 = gt_uint8.squeeze().numpy()  # (H,W,3) → 与单个代码一致
            
            # 图像增强（完全复刻单个代码的推理逻辑）
            enhanced_img, enhanced_path, pred_rgb_tensor = enhance_raw_batch(input_raw, enhance_model, base_name)
            if enhanced_img is None or pred_rgb_tensor is None:
                continue
            
            # 尺寸校验（确保增强图与GT尺寸一致）
            if gt_uint8.shape != enhanced_img.shape:
                print(f"警告：尺寸不匹配（GT: {gt_uint8.shape[:2]}, 增强图: {enhanced_img.shape[:2]}），跳过指标计算")
                continue
            
            # 计算指标（完全复刻单个正确代码的函数调用）
            psnr_val = calculate_psnr_train_style(gt_rgb, pred_rgb_tensor)  # 输入为(1,3,H,W)张量
            ssim_val = calculate_ssim(gt_uint8, enhanced_img)
            lpips_val = calculate_lpips(gt_uint8, enhanced_img, lpips_model)
            wbe_val = calculate_white_balance_error(gt_uint8, enhanced_img)
            
            # 记录结果
            results.append({
                "样本名称": base_name,
                "RAW文件路径": os.path.join(RAW_DIR, f"{base_name}{RAW_SUFFIX}"),
                "GT文件路径": os.path.join(RGB_DIR, f"{base_name}{RGB_SUFFIX}"),
                "增强图路径": enhanced_path,
                "PSNR (dB)": round(psnr_val, 4),
                "SSIM": round(ssim_val, 4),
                "LPIPS (越小越好)": round(lpips_val, 4),
                "白平衡误差 (WBE)": round(wbe_val, 4)
            })
            
            # 打印当前结果
            print(f"  PSNR: {psnr_val:.4f} dB | SSIM: {ssim_val:.4f} | LPIPS: {lpips_val:.4f} | WBE: {wbe_val:.4f}")
        
        except Exception as e:
            print(f"错误：处理样本 {base_name} 失败 - {str(e)}")
            continue
    
    # 保存Excel结果

    if results:
        df = pd.DataFrame(results)
        
        # 新增1：创建平均值行
        avg_row = {
            "样本名称": "平均值",
            "RAW文件路径": "",
            "GT文件路径": "",
            "增强图路径": "",
            "PSNR (dB)": round(df["PSNR (dB)"].mean(), 4),
            "SSIM": round(df["SSIM"].mean(), 4),
            "LPIPS (越小越好)": round(df["LPIPS (越小越好)"].mean(), 4),
            "白平衡误差 (WBE)": round(df["白平衡误差 (WBE)"].mean(), 4)
        }
        
        # 新增2：追加平均值行到DataFrame
        df_with_avg = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
        
        # 修改3：保存带平均值行的新DataFrame（原df改为df_with_avg）
        df_with_avg.to_excel(OUTPUT_EXCEL, index=False, engine="openpyxl")

        df.to_excel(OUTPUT_EXCEL, index=False, engine="openpyxl")
        
        # 统计摘要
        print("\n" + "="*60)
        print("批量处理完成！")
        print(f"✅ 成功处理 {len(results)}/{total_samples} 个样本")
        print(f"📊 评估结果已保存到：{OUTPUT_EXCEL}")
        print(f"🖼️  增强图已保存到：{ENHANCE_DIR}")
        print(f"   - 增强图前缀：{ENHANCED_PREFIX}")
        print(f"   - 保存格式：{SAVE_FORMAT.upper()}")
        
        print("\n【评估指标统计摘要】")
        print(f"平均 PSNR: {df['PSNR (dB)'].mean():.4f} ± {df['PSNR (dB)'].std():.4f} dB")
        print(f"平均 SSIM: {df['SSIM'].mean():.4f} ± {df['SSIM'].std():.4f}")
        print(f"平均 LPIPS: {df['LPIPS (越小越好)'].mean():.4f} ± {df['LPIPS (越小越好)'].std():.4f}")
        print(f"平均 WBE: {df['白平衡误差 (WBE)'].mean():.4f} ± {df['白平衡误差 (WBE)'].std():.4f}")
    else:
        print("\n❌ 没有成功处理任何样本！")

if __name__ == "__main__":
    # 依赖检查
    try:
        import pandas
        import openpyxl
        import skimage
        import lpips
    except ImportError as e:
        missing_pkg = str(e).split("'")[1]
        print(f"缺少依赖包：{missing_pkg}")
        print("请执行以下命令安装：")
        print("pip install pandas openpyxl scikit-image lpips torch torchvision opencv-python numpy")
        exit(1)
    
    main()

增强图将保存到：eval_dataset/extreme_dark/ENHANCE
Excel结果将保存到：eval_dataset/extreme_dark/enhanced_image_evaluation.xlsx
开始批量评估（完全复刻单个正确代码逻辑）
RAW文件目录：eval_dataset/extreme_dark/RAW
GT文件目录：eval_dataset/extreme_dark/RGB
增强图目录：eval_dataset/extreme_dark/ENHANCE

加载增强模型...
模型已加载到 cuda，权重路径：result/Freetech/weights/model_best.pth

初始化LPIPS模型（alex）...
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /environment/miniconda3/envs/eval_env/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth

加载评估数据集...
找到 4 组有效配对文件

开始处理 4 个样本...

[1/4] 处理样本：['19700101_09_05_32_1920_1280_113700']
增强图已保存：enhanced_['19700101_09_05_32_1920_1280_113700'].bmp
  PSNR: 27.5588 dB | SSIM: 0.9028 | LPIPS: 0.0984 | WBE: 0.0885

[2/4] 处理样本：['19700101_09_05_35_1920_1280_113790']
增强图已保存：enhanced_['19700101_09_05_35_1920_1280_113790'].bmp
  PSNR: 25.0716 dB | SSIM: 0.7443 | LPIPS: 0.2105 | WBE: 0.0422

[3/4] 处理样本：['19700101_09_53_04_1920_1280_199260']
增强图已保存：enhanced_['19700101_09_53_04_192

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import lpips
import torch
from torch.utils.data import DataLoader, Dataset
from typing import Dict, Tuple, List, Optional, Union
from model_freetech import RawFormer
import warnings
import imageio  # 新增：对齐Web版本的保存方式
warnings.filterwarnings("ignore")

# -------------------------- 配置参数（与单个正确代码完全一致） --------------------------
# 基础目录（daylight根目录）
BASE_DIR = "eval_dataset/test"
# 分目录配置（RAW和RGB文件分别存放）
RAW_DIR = os.path.join(BASE_DIR, "RAW")       # RAW文件路径：eval_dataset/daylight/RAW
RGB_DIR = os.path.join(BASE_DIR, "RGB")       # BMP文件路径：eval_dataset/daylight/RGB
ENHANCE_DIR = os.path.join(BASE_DIR, "ENHANCE")  # 增强图保存路径：eval_dataset/daylight/ENHANCE

# 输出配置
ENHANCED_PREFIX = "enhanced_"
OUTPUT_EXCEL = os.path.join(BASE_DIR, "enhanced_image_evaluation.xlsx")  # Excel保存到daylight目录

# 模型和设备配置（与单个正确代码一致）
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LPIPS_MODEL = "alex"
MODEL_SIZE = "S"  # 训练时的model_size
MODEL_DIM = 32 if MODEL_SIZE == "S" else 48 if MODEL_SIZE == "B" else 64
CHECKPOINT_PATH = "result/Freetech/weights/model_best.pth"  # 与单个正确代码一致

# 数据配置（与单个正确代码完全一致！）
RAW_WIDTH = 1920
RAW_HEIGHT = 1280
RGB_WIDTH = 1920
RGB_HEIGHT = 1280
RAW_HEADER_SIZE = 0  # 与单个正确代码一致
NORMALIZE_RANGE = (0, 1)  # 与单个正确代码一致

# 增强功能配置
RAW_SUFFIX = ".raw"  # RAW文件后缀
RGB_SUFFIX = ".bmp"  # GT文件后缀（BMP）
SAVE_FORMAT = "bmp"  # 增强图保存格式：仅保留BMP（与单个代码逻辑一致）

# ----------------------------------------------------------------

# 确保输出目录存在
os.makedirs(ENHANCE_DIR, exist_ok=True)
print(f"增强图将保存到：{ENHANCE_DIR}")
print(f"Excel结果将保存到：{OUTPUT_EXCEL}")

class RawRGB_Paired_SeparateDir(Dataset):
    """适配RAW和RGB分目录存储的数据集类（完全复刻单个正确代码的处理逻辑）"""
    def __init__(
        self,
        raw_dir: str,
        rgb_dir: str,
        raw_width: int,
        raw_height: int,
        raw_header_size: int = 0,
        rgb_width: int = 1920,
        rgb_height: int = 1280,
        normalize_range: Tuple[float, float] = (0, 1),
        raw_suffix: str = ".raw",
        rgb_suffix: str = ".bmp"
    ):
        self.raw_dir = raw_dir
        self.rgb_dir = rgb_dir
        self.raw_width = raw_width
        self.raw_height = raw_height
        self.raw_header_size = raw_header_size
        self.rgb_width = rgb_width
        self.rgb_height = rgb_height
        self.min_val, self.max_val = normalize_range
        self.raw_suffix = raw_suffix.lower()
        self.rgb_suffix = rgb_suffix.lower()

        # 扫描RAW目录下的所有RAW文件，并匹配RGB目录下的同名GT文件
        self.raw_rgb_pairs = self._scan_paired_files()
        if len(self.raw_rgb_pairs) == 0:
            raise ValueError(
                f"未找到配对的 {raw_suffix} 和 {rgb_suffix} 文件\n"
                f"检查目录：\n- RAW: {raw_dir}\n- RGB: {rgb_dir}\n"
                f"确保文件名完全一致（仅后缀不同）"
            )
        print(f"找到 {len(self.raw_rgb_pairs)} 组有效配对文件")

    def _scan_paired_files(self) -> List[Tuple[str, str]]:
        """扫描RAW目录，匹配RGB目录下的同名GT文件"""
        paired_files = []
        
        # 遍历RAW目录下的所有RAW文件
        for raw_filename in os.listdir(self.raw_dir):
            if raw_filename.lower().endswith(self.raw_suffix):
                # 提取基础文件名（不含后缀）
                base_name = os.path.splitext(raw_filename)[0]
                # 构建RGB目录下的GT文件路径
                rgb_filename = f"{base_name}{self.rgb_suffix}"
                rgb_path = os.path.join(self.rgb_dir, rgb_filename)
                raw_path = os.path.join(self.raw_dir, raw_filename)
                
                # 检查GT文件是否存在
                if os.path.exists(rgb_path):
                    paired_files.append((raw_path, rgb_path))
                else:
                    print(f"警告：跳过 {raw_filename} - RGB目录中未找到 {rgb_filename}")
        
        return paired_files

    def _load_raw(self, raw_path: str) -> Optional[np.ndarray]:
        """加载RAW文件（完全复刻单个正确代码的逻辑）"""
        try:
            with open(raw_path, 'rb') as f:
                f.seek(self.raw_header_size)
                raw_data = np.fromfile(f, dtype=np.uint16)
            # 确保数据量足够
            expected_size = self.raw_height * self.raw_width
            if len(raw_data) < expected_size:
                print(f"警告：RAW文件 {os.path.basename(raw_path)} 数据不完整（实际：{len(raw_data)}，期望：{expected_size}）")
                return None
            raw_img = raw_data[:expected_size].reshape(self.raw_height, self.raw_width)
            return raw_img.astype(np.float32)
        except Exception as e:
            print(f"错误：加载RAW文件 {os.path.basename(raw_path)} 失败 - {str(e)}")
            return None

    def _normalize_raw(self, raw_img: np.ndarray) -> np.ndarray:
        """【核心修改1】完全对齐Web版本的RAW预处理逻辑（强制黑电平处理）"""
        ap = 100
        black_level = 512
        max_raw = 16383
        raw_processed = (np.maximum(raw_img.astype(np.float32) - black_level, 0) / (max_raw - black_level)) * ap
        return raw_processed

    def _load_rgb(self, rgb_path: str) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        """加载GT图（完全复刻单个正确代码的逻辑）"""
        try:
            bmp_img = cv2.imread(rgb_path, cv2.IMREAD_COLOR)
            if bmp_img is None:
                print(f"警告：GT文件 {os.path.basename(rgb_path)} 损坏或无法读取")
                return None, None
            
            bmp_img = cv2.cvtColor(bmp_img, cv2.COLOR_BGR2RGB)
            # 调整尺寸到训练时的大小（与单个正确代码一致）
            if bmp_img.shape[:2] != (self.rgb_height, self.rgb_width):
                bmp_img = cv2.resize(bmp_img, (self.rgb_width, self.rgb_height), interpolation=cv2.INTER_LINEAR)
            
            bmp_uint8 = bmp_img.astype(np.uint8)
            bmp_norm = bmp_uint8.astype(np.float32) / 255.0  # 与单个正确代码一致：(0,1)范围
            return bmp_norm, bmp_uint8
        except Exception as e:
            print(f"错误：加载GT文件 {os.path.basename(rgb_path)} 失败 - {str(e)}")
            return None, None

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, np.ndarray, str]:
        """返回 (input_raw, gt_rgb, gt_uint8, base_name) - 完全复刻单个正确代码的张量格式"""
        raw_path, rgb_path = self.raw_rgb_pairs[idx]
        base_name = os.path.splitext(os.path.basename(raw_path))[0]

        # 加载数据（增加有效性检查）
        raw_img = self._load_raw(raw_path)
        bmp_norm, bmp_uint8 = self._load_rgb(rgb_path)
        if raw_img is None or bmp_norm is None or bmp_uint8 is None:
            raise ValueError(f"样本 {base_name} 数据加载失败")

        # RAW归一化（【核心修改】已对齐Web版本）
        raw_norm = self._normalize_raw(raw_img)

        # 转换为张量（严格复刻单个正确代码的格式）
        input_raw = torch.from_numpy(raw_norm).unsqueeze(0).float()  # (1, H, W) - 与单个代码一致
        gt_rgb = torch.from_numpy(bmp_norm.transpose(2, 0, 1)).float()  # (3, H, W) - 与单个代码一致

        return input_raw, gt_rgb, bmp_uint8, base_name

    def __len__(self) -> int:
        return len(self.raw_rgb_pairs)

# -------------------------- 模型加载（完全复刻单个正确代码） --------------------------
def load_enhance_model() -> RawFormer:
    num_virtual_cams = 5  # 与单个正确代码一致
    pretrain_mode = False
    model = RawFormer(
        dim=MODEL_DIM,
        use_rep_nr=True,
        num_virtual_cams=num_virtual_cams,
        pretrain_mode=pretrain_mode
    )
    # 加载权重（与单个正确代码一致）
    try:
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
        state_dict = checkpoint["state_dict"]
        new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(new_state_dict, strict=False)
        model = model.to(DEVICE)
        model.eval()
        print(f"模型已加载到 {DEVICE}，权重路径：{CHECKPOINT_PATH}")
        return model
    except Exception as e:
        print(f"错误：加载模型失败 - {str(e)}")
        raise SystemExit(1)

# -------------------------- 数据加载（适配分目录，保持单个代码的处理逻辑） --------------------------
def create_eval_dataset() -> Tuple[DataLoader, List[str]]:
    # 检查RAW和RGB目录是否存在
    if not os.path.exists(RAW_DIR):
        raise FileNotFoundError(f"RAW目录不存在：{RAW_DIR}")
    if not os.path.exists(RGB_DIR):
        raise FileNotFoundError(f"RGB目录不存在：{RGB_DIR}")
    
    dataset = RawRGB_Paired_SeparateDir(
        raw_dir=RAW_DIR,
        rgb_dir=RGB_DIR,
        raw_width=RAW_WIDTH,
        raw_height=RAW_HEIGHT,
        raw_header_size=RAW_HEADER_SIZE,
        rgb_width=RGB_WIDTH,
        rgb_height=RGB_HEIGHT,
        normalize_range=NORMALIZE_RANGE,
        raw_suffix=RAW_SUFFIX,
        rgb_suffix=RGB_SUFFIX
    )
    dataloader = DataLoader(
        dataset,
        batch_size=1,  # 保持batch=1，与单个代码一致
        shuffle=False,
        num_workers=0,  # 单线程，避免数据错乱（与单个代码一致）
        pin_memory=True if torch.cuda.is_available() else False,
        drop_last=False
    )
    base_names = [os.path.splitext(os.path.basename(f[0]))[0] for f in dataset.raw_rgb_pairs]
    return dataloader, base_names

# -------------------------- 图像增强（【核心修改2】完全对齐Web版本的推理/后处理逻辑） --------------------------
def enhance_raw_batch(
    input_raw: torch.Tensor, 
    model: RawFormer, 
    base_name: str,
    save_dir: str = ENHANCE_DIR
) -> Tuple[Optional[np.ndarray], Optional[str], Optional[torch.Tensor]]:
    """增强RAW图像（完全对齐Web版本的推理逻辑），保存到ENHANCE目录"""
    try:
        # 模型推理（与Web版本完全一致）
        with torch.no_grad():
            pred_rgb_tensor = model(input_raw, cam_id=0)  # 保留batch维度：(1,3,H,W)
            pred_rgb_tensor = torch.clamp(pred_rgb_tensor, 0, 1)  # 与Web版本一致
        
        # 后处理（完全对齐Web版本：squeeze()无参数 + 转置(1,2,0) + imageio保存）
        pred_rgb_np = pred_rgb_tensor.cpu().numpy()  # (1,3,H,W)
        pred_rgb_np = pred_rgb_np.squeeze()  # Web版本用squeeze()无参数 → (3,H,W)
        pred_rgb_np = pred_rgb_np.transpose((1, 2, 0))  # (H,W,3)
        pred_rgb_np = (pred_rgb_np * 255).astype(np.uint8)  # 与Web版本一致的数值范围转换
        
        # 保存增强图（Web版本用imageio，避免OpenCV的BGR/RGB转换）
        enhanced_path = os.path.join(save_dir, f"{ENHANCED_PREFIX}{base_name}.bmp")
        imageio.imwrite(enhanced_path, pred_rgb_np)  # 直接保存RGB，无需通道转换
        
        print(f"增强图已保存：{os.path.basename(enhanced_path)}")
        return pred_rgb_np, enhanced_path, pred_rgb_tensor  # 返回原始维度的张量 (1,3,H,W)
    except Exception as e:
        print(f"错误：增强图像 {base_name} 失败 - {str(e)}")
        return None, None, None

# -------------------------- 指标计算（完全复刻单个正确代码的函数） --------------------------
def calculate_psnr_train_style(gt_rgb_tensor: torch.Tensor, pred_rgb_tensor: torch.Tensor) -> float:
    """完全复刻单个正确代码的PSNR计算逻辑（适配Web版本的维度）"""
    # 验证输入维度（确保与Web版本一致）
    assert gt_rgb_tensor.shape == pred_rgb_tensor.shape, f"维度不匹配：gt={gt_rgb_tensor.shape}, pred={pred_rgb_tensor.shape}"
    assert gt_rgb_tensor.shape[0] == 1, f"batch维度错误：{gt_rgb_tensor.shape[0]}"
    assert gt_rgb_tensor.shape[1] == 3, f"通道维度错误：{gt_rgb_tensor.shape[1]}"
    
    # 适配Web版本的维度：先squeeze(0)再处理
    gt_np = gt_rgb_tensor.squeeze(0).cpu().numpy().transpose(1, 2, 0)  # (H,W,3)
    pred_np = pred_rgb_tensor.squeeze(0).detach().cpu().numpy().transpose(1, 2, 0)  # (H,W,3)
    pred_np = np.clip(pred_np, 0.0, 1.0)
    
    # 转uint8（与单个正确代码一致）
    gt_uint8 = (gt_np * 255).astype(np.uint8)
    pred_uint8 = (pred_np * 255).astype(np.uint8)
    
    psnr_value = psnr(gt_uint8, pred_uint8, data_range=255)
    return psnr_value

def calculate_ssim(gt_uint8: np.ndarray, pred_uint8: np.ndarray) -> float:
    """完全复刻单个正确代码的SSIM计算"""
    return ssim(
        gt_uint8, pred_uint8,
        data_range=255,
        channel_axis=2,
        gaussian_weights=True,
        sigma=1.5
    )

def calculate_lpips(gt_uint8: np.ndarray, pred_uint8: np.ndarray, lpips_model) -> float:
    """完全复刻单个正确代码的LPIPS计算"""
    def preprocess(img: np.ndarray) -> torch.Tensor:
        img_norm = (img / 255.0) * 2 - 1  # 0-255 → [-1,1]（与单个代码一致）
        return torch.from_numpy(img_norm.transpose((2, 0, 1))).float().unsqueeze(0).to(DEVICE)
    
    gt_tensor = preprocess(gt_uint8)
    pred_tensor = preprocess(pred_uint8)
    
    with torch.no_grad():
        dist = lpips_model(gt_tensor, pred_tensor).item()
    return dist

def calculate_white_balance_error(gt_uint8: np.ndarray, pred_uint8: np.ndarray) -> float:
    """完全复刻单个正确代码的WBE计算"""
    gt = gt_uint8.astype(np.float32) / 255.0
    pred = pred_uint8.astype(np.float32) / 255.0
    
    # 计算RGB通道均值
    gt_r, gt_g, gt_b = np.mean(gt[..., 0]), np.mean(gt[..., 1]), np.mean(gt[..., 2])
    pred_r, pred_g, pred_b = np.mean(pred[..., 0]), np.mean(pred[..., 1]), np.mean(pred[..., 2])
    
    # 避免除以零
    gt_g = max(gt_g, 1e-6)
    pred_g = max(pred_g, 1e-6)
    
    # 计算R/G、B/G比值误差
    return abs((pred_r/pred_g) - (gt_r/gt_g)) + abs((pred_b/pred_g) - (gt_b/gt_g))

# -------------------------- 主流程（完全复刻单个正确代码的逻辑，仅适配增强函数修改） --------------------------
def main():
    print("="*60)
    print(f"开始批量评估（RAW处理逻辑对齐Web版本）")
    print(f"RAW文件目录：{RAW_DIR}")
    print(f"GT文件目录：{RGB_DIR}")
    print(f"增强图目录：{ENHANCE_DIR}")
    print("="*60)

    # 加载模型（与单个正确代码一致）
    print("\n加载增强模型...")
    enhance_model = load_enhance_model()
    
    # 初始化LPIPS（与单个正确代码一致）
    print(f"\n初始化LPIPS模型（{LPIPS_MODEL}）...")
    try:
        lpips_model = lpips.LPIPS(net=LPIPS_MODEL).to(DEVICE)
        lpips_model.eval()
    except Exception as e:
        print(f"错误：初始化LPIPS失败 - {str(e)}")
        return
    
    # 加载数据集（分目录结构，保持单个代码的处理逻辑）
    print("\n加载评估数据集...")
    try:
        eval_dataloader, base_names = create_eval_dataset()
    except (ValueError, FileNotFoundError) as e:
        print(f"错误：{e}")
        return
    
    # 批量处理（逐个样本处理，完全复刻单个代码的逻辑）
    results = []
    total_samples = len(eval_dataloader)
    print(f"\n开始处理 {total_samples} 个样本...")
    print("="*60)
    
    for idx, (input_raw, gt_rgb, gt_uint8, base_name) in enumerate(eval_dataloader, 1):
        print(f"\n[{idx}/{total_samples}] 处理样本：{base_name}")
        
        try:
            # 数据移到设备（与单个正确代码一致，保持batch维度）
            input_raw = input_raw.to(DEVICE)  # (1,1,H,W)
            gt_rgb = gt_rgb.to(DEVICE)        # (1,3,H,W) → 与单个代码一致
            gt_uint8 = gt_uint8.squeeze().numpy()  # (H,W,3) → 与单个代码一致
            
            # 图像增强（对齐Web版本的推理逻辑）
            enhanced_img, enhanced_path, pred_rgb_tensor = enhance_raw_batch(input_raw, enhance_model, base_name)
            if enhanced_img is None or pred_rgb_tensor is None:
                continue
            
            # 尺寸校验（确保增强图与GT尺寸一致）
            if gt_uint8.shape != enhanced_img.shape:
                print(f"警告：尺寸不匹配（GT: {gt_uint8.shape[:2]}, 增强图: {enhanced_img.shape[:2]}），跳过指标计算")
                continue
            
            # 计算指标（完全复刻单个正确代码的函数调用）
            psnr_val = calculate_psnr_train_style(gt_rgb, pred_rgb_tensor)  # 输入为(1,3,H,W)张量
            ssim_val = calculate_ssim(gt_uint8, enhanced_img)
            lpips_val = calculate_lpips(gt_uint8, enhanced_img, lpips_model)
            wbe_val = calculate_white_balance_error(gt_uint8, enhanced_img)
            
            # 记录结果
            results.append({
                "样本名称": base_name,
                "RAW文件路径": os.path.join(RAW_DIR, f"{base_name}{RAW_SUFFIX}"),
                "GT文件路径": os.path.join(RGB_DIR, f"{base_name}{RGB_SUFFIX}"),
                "增强图路径": enhanced_path,
                "PSNR (dB)": round(psnr_val, 4),
                "SSIM": round(ssim_val, 4),
                "LPIPS (越小越好)": round(lpips_val, 4),
                "白平衡误差 (WBE)": round(wbe_val, 4)
            })
            
            # 打印当前结果
            print(f"  PSNR: {psnr_val:.4f} dB | SSIM: {ssim_val:.4f} | LPIPS: {lpips_val:.4f} | WBE: {wbe_val:.4f}")
        
        except Exception as e:
            print(f"错误：处理样本 {base_name} 失败 - {str(e)}")
            continue
    
    # 保存Excel结果
    if results:
        df = pd.DataFrame(results)
        
        # 新增1：创建平均值行
        avg_row = {
            "样本名称": "平均值",
            "RAW文件路径": "",
            "GT文件路径": "",
            "增强图路径": "",
            "PSNR (dB)": round(df["PSNR (dB)"].mean(), 4),
            "SSIM": round(df["SSIM"].mean(), 4),
            "LPIPS (越小越好)": round(df["LPIPS (越小越好)"].mean(), 4),
            "白平衡误差 (WBE)": round(df["白平衡误差 (WBE)"].mean(), 4)
        }
        
        # 新增2：追加平均值行到DataFrame
        df_with_avg = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)
        
        # 修改3：保存带平均值行的新DataFrame（原df改为df_with_avg）
        df_with_avg.to_excel(OUTPUT_EXCEL, index=False, engine="openpyxl")

        # 统计摘要
        print("\n" + "="*60)
        print("批量处理完成！")
        print(f"✅ 成功处理 {len(results)}/{total_samples} 个样本")
        print(f"📊 评估结果已保存到：{OUTPUT_EXCEL}")
        print(f"🖼️  增强图已保存到：{ENHANCE_DIR}")
        print(f"   - 增强图前缀：{ENHANCED_PREFIX}")
        print(f"   - 保存格式：{SAVE_FORMAT.upper()}")
        
        print("\n【评估指标统计摘要】")
        print(f"平均 PSNR: {df['PSNR (dB)'].mean():.4f} ± {df['PSNR (dB)'].std():.4f} dB")
        print(f"平均 SSIM: {df['SSIM'].mean():.4f} ± {df['SSIM'].std():.4f}")
        print(f"平均 LPIPS: {df['LPIPS (越小越好)'].mean():.4f} ± {df['LPIPS (越小越好)'].std():.4f}")
        print(f"平均 WBE: {df['白平衡误差 (WBE)'].mean():.4f} ± {df['白平衡误差 (WBE)'].std():.4f}")
    else:
        print("\n❌ 没有成功处理任何样本！")

if __name__ == "__main__":
    # 依赖检查（新增imageio）
    try:
        import pandas
        import openpyxl
        import skimage
        import lpips
        import imageio  # 新增检查
    except ImportError as e:
        missing_pkg = str(e).split("'")[1]
        print(f"缺少依赖包：{missing_pkg}")
        print("请执行以下命令安装：")
        print("pip install pandas openpyxl scikit-image lpips torch torchvision opencv-python numpy imageio")
        exit(1)
    
    main()

增强图将保存到：eval_dataset/test/ENHANCE
Excel结果将保存到：eval_dataset/test/enhanced_image_evaluation.xlsx
开始批量评估（RAW处理逻辑对齐Web版本）
RAW文件目录：eval_dataset/test/RAW
GT文件目录：eval_dataset/test/RGB
增强图目录：eval_dataset/test/ENHANCE

加载增强模型...
模型已加载到 cuda，权重路径：result/Freetech/weights/model_best.pth

初始化LPIPS模型（alex）...
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /environment/miniconda3/envs/eval_env/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth

加载评估数据集...
找到 4 组有效配对文件

开始处理 4 个样本...

[1/4] 处理样本：['20251208-005734']
增强图已保存：enhanced_['20251208-005734'].bmp
  PSNR: 21.9845 dB | SSIM: 0.2845 | LPIPS: 0.5051 | WBE: 0.0638

[2/4] 处理样本：['20251208-005755']
增强图已保存：enhanced_['20251208-005755'].bmp
  PSNR: 24.0431 dB | SSIM: 0.3989 | LPIPS: 0.3492 | WBE: 12.4210

[3/4] 处理样本：['20251208-005828']
增强图已保存：enhanced_['20251208-005828'].bmp
  PSNR: 29.5703 dB | SSIM: 0.8087 | LPIPS: 0.1925 | WBE: 1.8574

[4/4] 处理样本：['20251208-005814']
增强图已保存：enhanced_['20251208-005814']

Test Model Size and Model Memory

In [5]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import lpips
import torch
from torch.utils.data import DataLoader, Dataset
from typing import Dict, Tuple, List, Optional, Union
# 导入你的 RawFormer 模型（确保 model_freetech.py 在当前目录，或直接复制上面的 RawFormer 代码到当前文件）
from model_freetech import RawFormer  
import warnings
import imageio
warnings.filterwarnings("ignore")

# -------------------------- 原有配置参数 --------------------------
BASE_DIR = "eval_dataset/daylight"
RAW_DIR = os.path.join(BASE_DIR, "RAW")
RGB_DIR = os.path.join(BASE_DIR, "RGB")
ENHANCE_DIR = os.path.join(BASE_DIR, "ENHANCE")
ENHANCED_PREFIX = "enhanced_"
OUTPUT_EXCEL = os.path.join(BASE_DIR, "enhanced_image_evaluation.xlsx")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LPIPS_MODEL = "alex"
MODEL_SIZE = "S"  # 可切换为"B"测试
MODEL_DIM = 32 if MODEL_SIZE == "S" else 48 if MODEL_SIZE == "B" else 64
CHECKPOINT_PATH = "result/Freetech/weights/model_best.pth"

# 创建增强图保存目录
os.makedirs(ENHANCE_DIR, exist_ok=True)

# -------------------------- 新增：模型大小/内存计算函数 --------------------------
def calculate_model_size(model: torch.nn.Module, dtype: torch.dtype = torch.float32) -> Dict[str, float]:
    """计算模型的磁盘大小（仅参数）"""
    total_params = sum(p.numel() for p in model.parameters())
    dtype_bytes = torch.finfo(dtype).bits // 8
    total_bytes = total_params * dtype_bytes
    model_size_mb = total_bytes / (1024 * 1024)
    model_size_gb = total_bytes / (1024 * 1024 * 1024)
    
    return {
        "total_params": total_params,
        "model_size_mb": model_size_mb,
        "model_size_gb": model_size_gb,
        "dtype": dtype
    }

def calculate_model_memory(
    model: torch.nn.Module, 
    input_shape: Tuple[int, int, int, int] = (1, 1, 1024, 1024)  # RAW输入：(B, 1, H, W)，匹配你的 inp_channels=1
) -> Dict[str, float]:
    """计算模型运行时内存/显存占用"""
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    
    # 1. 仅参数的显存占用
    param_memory_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    param_memory_mb = param_memory_bytes / (1024 * 1024)
    
    # 2. 模拟推理，计算运行时/峰值显存
    if DEVICE.type == "cuda":
        initial_mem = torch.cuda.memory_allocated(DEVICE) / (1024 * 1024)
        # 生成测试输入（匹配你的 RAW 输入：单通道）
        test_input = torch.randn(input_shape, device=DEVICE, dtype=torch.float32)
        # 前向推理（无梯度，cam_id=0 是默认值）
        with torch.no_grad():
            _ = model(test_input, cam_id=0)
        # 运行时/峰值显存
        runtime_mem = torch.cuda.memory_allocated(DEVICE) / (1024 * 1024)
        peak_mem = torch.cuda.max_memory_allocated(DEVICE) / (1024 * 1024)
        torch.cuda.reset_peak_memory_stats(DEVICE)
    else:
        runtime_mem = param_memory_mb
        peak_mem = param_memory_mb
    
    return {
        "param_memory_mb": param_memory_mb,
        "runtime_memory_mb": runtime_mem,
        "peak_memory_mb": peak_mem,
        "input_shape": input_shape
    }

def calculate_ckpt_file_size(ckpt_path: str) -> float:
    """计算模型权重文件的磁盘大小（MB）"""
    if os.path.exists(ckpt_path):
        return os.path.getsize(ckpt_path) / (1024 * 1024)
    else:
        print(f"⚠️ 权重文件不存在: {ckpt_path}")
        return 0.0

# -------------------------- 修正：加载 RawFormer 模型（完全匹配你的参数） --------------------------
def load_rawformer_model() -> RawFormer:
    """加载RawFormer模型（严格匹配你的 __init__ 参数）"""
    # 你的 RawFormer 初始化参数（从源码中提取）
    model = RawFormer(
        inp_channels=1,          # RAW图像是单通道，匹配你的默认值
        out_channels=3,          # 输出RGB三通道，匹配你的默认值
        dim=MODEL_DIM,           # S=32/B=48，对应你的配置
        num_heads=[8,8,8,8],     # 你的默认值，可根据训练时的参数调整
        ffn_expansion_factor=2,  # 你的默认值
        use_rep_nr=True,         # 启用 RepNR 模块，匹配你的默认值
        num_virtual_cams=5,      # 虚拟相机数，匹配你的默认值
        pretrain_mode=False      # 非预训练模式，匹配你的默认值
    ).to(DEVICE)
    
    # 加载权重（兼容 state_dict 或直接保存的权重）
    if os.path.exists(CHECKPOINT_PATH):
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
        state_dict = checkpoint.get("state_dict", checkpoint)
        # 处理可能的 key 前缀（如 'module.'）
        if list(state_dict.keys())[0].startswith("module."):
            state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(state_dict, strict=True)
    model.eval()  # 推理模式，不修改参数
    return model

# -------------------------- 主执行逻辑 --------------------------
if __name__ == "__main__":
    # 1. 加载模型（无参数不匹配报错）
    print("===== 加载RawFormer模型 =====")
    model = load_rawformer_model()
    print("✅ 模型加载成功！")
    
    # 2. 计算 Model Size（磁盘大小）
    print("\n===== Model Size（模型磁盘大小） =====")
    size_info = calculate_model_size(model, dtype=torch.float32)
    print(f"总参数数量: {size_info['total_params']:,}")
    print(f"理论磁盘大小 (float32): {size_info['model_size_mb']:.2f} MB / {size_info['model_size_gb']:.4f} GB")
    ckpt_size = calculate_ckpt_file_size(CHECKPOINT_PATH)
    print(f"权重文件实际大小: {ckpt_size:.2f} MB")
    
    # 3. 计算 Model Memory（运行时内存/显存）
    print("\n===== Model Memory（运行时内存/显存） =====")
    # 输入形状：(批量大小1, 单通道, 1024x1024)，可根据你的实际RAW图像尺寸调整
    mem_info = calculate_model_memory(model, input_shape=(1, 1, 1024, 1024))
    print(f"仅参数显存占用: {mem_info['param_memory_mb']:.2f} MB")
    print(f"推理时显存占用: {mem_info['runtime_memory_mb']:.2f} MB")
    print(f"推理峰值显存占用: {mem_info['peak_memory_mb']:.2f} MB")
    print(f"测试输入形状: {mem_info['input_shape']}")
    
    # 4. （可选）训练态内存（参数+梯度+优化器）
    print("\n===== 训练态总内存（可选） =====")
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    total_train_mem = 0.0
    # 参数+梯度
    for p in model.parameters():
        total_train_mem += p.numel() * p.element_size()
        if p.grad is not None:
            total_train_mem += p.grad.numel() * p.grad.element_size()
    # 优化器状态（Adam的m/v）
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                total_train_mem += v.numel() * v.element_size()
    total_train_mem_mb = total_train_mem / (1024 * 1024)
    print(f"训练态（参数+梯度+Adam优化器）总内存: {total_train_mem_mb:.2f} MB")

===== 加载RawFormer模型 =====
✅ 模型加载成功！

===== Model Size（模型磁盘大小） =====
总参数数量: 5,363,492
理论磁盘大小 (float32): 20.46 MB / 0.0200 GB
权重文件实际大小: 54.17 MB

===== Model Memory（运行时内存/显存） =====
仅参数显存占用: 20.46 MB
推理时显存占用: 44.86 MB
推理峰值显存占用: 1388.33 MB
测试输入形状: (1, 1, 1024, 1024)

===== 训练态总内存（可选） =====
训练态（参数+梯度+Adam优化器）总内存: 20.46 MB


加入噪声

In [9]:
import os
import cv2
import numpy as np
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import lpips
import torch
from torch.utils.data import DataLoader, Dataset
from typing import Dict, Tuple, List, Optional, Union
from model_freetech import RawFormer
import warnings
import imageio
warnings.filterwarnings("ignore")

# -------------------------- 配置参数 --------------------------
# 基础目录
BASE_DIR = "eval_dataset/test"
RAW_DIR = os.path.join(BASE_DIR, "RAW")
RGB_DIR = os.path.join(BASE_DIR, "RGB")
ENHANCE_DIR = os.path.join(BASE_DIR, "ENHANCE")
NOISY_ENHANCE_DIR = os.path.join(BASE_DIR, "NOISY_ENHANCE")  # 噪声增强图保存目录

# 输出配置
ENHANCED_PREFIX = "enhanced_"
NOISY_ENHANCED_PREFIX = "noisy_enhanced_"
OUTPUT_EXCEL = os.path.join(BASE_DIR, "enhanced_image_evaluation.xlsx")
NOISE_TEST_EXCEL = os.path.join(BASE_DIR, "noise_performance_test.xlsx")  # 噪声测试结果

# 模型和设备配置
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LPIPS_MODEL = "alex"
MODEL_SIZE = "S"
MODEL_DIM = 32 if MODEL_SIZE == "S" else 48 if MODEL_SIZE == "B" else 64
CHECKPOINT_PATH = "result/Freetech/weights/model_best.pth"

# 数据配置
RAW_WIDTH = 1920
RAW_HEIGHT = 1280
RGB_WIDTH = 1920
RGB_HEIGHT = 1280
RAW_HEADER_SIZE = 0
NORMALIZE_RANGE = (0, 1)

# 增强功能配置
RAW_SUFFIX = ".raw"
RGB_SUFFIX = ".bmp"
SAVE_FORMAT = "bmp"

# 噪声配置
NOISE_TYPES = ["gaussian", "poisson"]  # 要测试的噪声类型
GAUSSIAN_NOISE_STD = [0.01, 0.03, 0.05]  # 高斯噪声标准差（归一化后）
POISSON_NOISE_SCALE = [50, 100, 200]     # 泊松噪声缩放因子

# ----------------------------------------------------------------

# 确保输出目录存在
os.makedirs(ENHANCE_DIR, exist_ok=True)
os.makedirs(NOISY_ENHANCE_DIR, exist_ok=True)
print(f"干净增强图保存到：{ENHANCE_DIR}")
print(f"噪声增强图保存到：{NOISY_ENHANCE_DIR}")
print(f"基础评估结果保存到：{OUTPUT_EXCEL}")
print(f"噪声测试结果保存到：{NOISE_TEST_EXCEL}")

class RawRGB_Paired_SeparateDir(Dataset):
    """适配RAW和RGB分目录存储的数据集类"""
    def __init__(
        self,
        raw_dir: str,
        rgb_dir: str,
        raw_width: int,
        raw_height: int,
        raw_header_size: int = 0,
        rgb_width: int = 1920,
        rgb_height: int = 1280,
        normalize_range: Tuple[float, float] = (0, 1),
        raw_suffix: str = ".raw",
        rgb_suffix: str = ".bmp",
        add_noise: bool = False,
        noise_type: str = "gaussian",
        noise_param: float = 0.01
    ):
        self.raw_dir = raw_dir
        self.rgb_dir = rgb_dir
        self.raw_width = raw_width
        self.raw_height = raw_height
        self.raw_header_size = raw_header_size
        self.rgb_width = rgb_width
        self.rgb_height = rgb_height
        self.min_val, self.max_val = normalize_range
        self.raw_suffix = raw_suffix.lower()
        self.rgb_suffix = rgb_suffix.lower()
        
        # 噪声相关参数
        self.add_noise = add_noise
        self.noise_type = noise_type.lower()
        self.noise_param = noise_param

        # 扫描配对文件
        self.raw_rgb_pairs = self._scan_paired_files()
        if len(self.raw_rgb_pairs) == 0:
            raise ValueError(
                f"未找到配对的 {raw_suffix} 和 {rgb_suffix} 文件\n"
                f"检查目录：\n- RAW: {raw_dir}\n- RGB: {rgb_dir}\n"
                f"确保文件名完全一致（仅后缀不同）"
            )
        print(f"找到 {len(self.raw_rgb_pairs)} 组有效配对文件")

    def _scan_paired_files(self) -> List[Tuple[str, str]]:
        """扫描RAW目录，匹配RGB目录下的同名GT文件"""
        paired_files = []
        
        for raw_filename in os.listdir(self.raw_dir):
            if raw_filename.lower().endswith(self.raw_suffix):
                base_name = os.path.splitext(raw_filename)[0]
                rgb_filename = f"{base_name}{self.rgb_suffix}"
                rgb_path = os.path.join(self.rgb_dir, rgb_filename)
                raw_path = os.path.join(self.raw_dir, raw_filename)
                
                if os.path.exists(rgb_path):
                    paired_files.append((raw_path, rgb_path))
                else:
                    print(f"警告：跳过 {raw_filename} - RGB目录中未找到 {rgb_filename}")
        
        return paired_files

    def _add_noise_to_raw(self, raw_img: np.ndarray) -> np.ndarray:
        """为RAW图像添加噪声"""
        # 保存原始数据范围
        min_val = np.min(raw_img)
        max_val = np.max(raw_img)
        
        # 归一化到0-1范围添加噪声
        raw_norm = (raw_img - min_val) / (max_val - min_val)
        
        if self.noise_type == "gaussian":
            # 高斯噪声
            noise = np.random.normal(0, self.noise_param, raw_norm.shape)
            raw_noisy = raw_norm + noise
        elif self.noise_type == "poisson":
            # 泊松噪声
            # 泊松噪声需要非负整数输入，先缩放再转换
            raw_scaled = raw_norm * self.noise_param
            raw_noisy = np.random.poisson(raw_scaled) / self.noise_param
        else:
            raise ValueError(f"不支持的噪声类型：{self.noise_type}")
        
        # 裁剪到0-1范围并恢复原始范围
        raw_noisy = np.clip(raw_noisy, 0, 1)
        raw_noisy = raw_noisy * (max_val - min_val) + min_val
        
        # 确保数据类型一致
        return raw_noisy.astype(raw_img.dtype)

    def _load_raw(self, raw_path: str) -> Optional[np.ndarray]:
        """加载RAW文件并可选添加噪声"""
        try:
            with open(raw_path, 'rb') as f:
                f.seek(self.raw_header_size)
                raw_data = np.fromfile(f, dtype=np.uint16)
            
            expected_size = self.raw_height * self.raw_width
            if len(raw_data) < expected_size:
                print(f"警告：RAW文件 {os.path.basename(raw_path)} 数据不完整（实际：{len(raw_data)}，期望：{expected_size}）")
                return None
            
            raw_img = raw_data[:expected_size].reshape(self.raw_height, self.raw_width)
            
            # 添加噪声
            if self.add_noise:
                raw_img = self._add_noise_to_raw(raw_img)
            
            return raw_img.astype(np.float32)
        except Exception as e:
            print(f"错误：加载RAW文件 {os.path.basename(raw_path)} 失败 - {str(e)}")
            return None

    def _normalize_raw(self, raw_img: np.ndarray) -> np.ndarray:
        """RAW预处理逻辑"""
        ap = 100
        black_level = 512
        max_raw = 16383
        raw_processed = (np.maximum(raw_img.astype(np.float32) - black_level, 0) / (max_raw - black_level)) * ap
        return raw_processed

    def _load_rgb(self, rgb_path: str) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
        """加载GT图"""
        try:
            bmp_img = cv2.imread(rgb_path, cv2.IMREAD_COLOR)
            if bmp_img is None:
                print(f"警告：GT文件 {os.path.basename(rgb_path)} 损坏或无法读取")
                return None, None
            
            bmp_img = cv2.cvtColor(bmp_img, cv2.COLOR_BGR2RGB)
            if bmp_img.shape[:2] != (self.rgb_height, self.rgb_width):
                bmp_img = cv2.resize(bmp_img, (self.rgb_width, self.rgb_height), interpolation=cv2.INTER_LINEAR)
            
            bmp_uint8 = bmp_img.astype(np.uint8)
            bmp_norm = bmp_uint8.astype(np.float32) / 255.0
            return bmp_norm, bmp_uint8
        except Exception as e:
            print(f"错误：加载GT文件 {os.path.basename(rgb_path)} 失败 - {str(e)}")
            return None, None

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, np.ndarray, str]:
        """返回 (input_raw, gt_rgb, gt_uint8, base_name)"""
        raw_path, rgb_path = self.raw_rgb_pairs[idx]
        base_name = os.path.splitext(os.path.basename(raw_path))[0]

        raw_img = self._load_raw(raw_path)
        bmp_norm, bmp_uint8 = self._load_rgb(rgb_path)
        if raw_img is None or bmp_norm is None or bmp_uint8 is None:
            raise ValueError(f"样本 {base_name} 数据加载失败")

        raw_norm = self._normalize_raw(raw_img)

        input_raw = torch.from_numpy(raw_norm).unsqueeze(0).float()
        gt_rgb = torch.from_numpy(bmp_norm.transpose(2, 0, 1)).float()

        return input_raw, gt_rgb, bmp_uint8, base_name

    def __len__(self) -> int:
        return len(self.raw_rgb_pairs)

# -------------------------- 模型加载 --------------------------
def load_enhance_model() -> RawFormer:
    num_virtual_cams = 5
    pretrain_mode = False
    model = RawFormer(
        dim=MODEL_DIM,
        use_rep_nr=True,
        num_virtual_cams=num_virtual_cams,
        pretrain_mode=pretrain_mode
    )
    try:
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
        state_dict = checkpoint["state_dict"]
        new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
        model.load_state_dict(new_state_dict, strict=False)
        model = model.to(DEVICE)
        model.eval()
        print(f"模型已加载到 {DEVICE}，权重路径：{CHECKPOINT_PATH}")
        return model
    except Exception as e:
        print(f"错误：加载模型失败 - {str(e)}")
        raise SystemExit(1)

# -------------------------- 数据加载 --------------------------
def create_eval_dataset(
    add_noise: bool = False,
    noise_type: str = "gaussian",
    noise_param: float = 0.01
) -> Tuple[DataLoader, List[str]]:
    """创建数据集加载器（支持噪声配置）"""
    if not os.path.exists(RAW_DIR):
        raise FileNotFoundError(f"RAW目录不存在：{RAW_DIR}")
    if not os.path.exists(RGB_DIR):
        raise FileNotFoundError(f"RGB目录不存在：{RGB_DIR}")
    
    dataset = RawRGB_Paired_SeparateDir(
        raw_dir=RAW_DIR,
        rgb_dir=RGB_DIR,
        raw_width=RAW_WIDTH,
        raw_height=RAW_HEIGHT,
        raw_header_size=RAW_HEADER_SIZE,
        rgb_width=RGB_WIDTH,
        rgb_height=RGB_HEIGHT,
        normalize_range=NORMALIZE_RANGE,
        raw_suffix=RAW_SUFFIX,
        rgb_suffix=RGB_SUFFIX,
        add_noise=add_noise,
        noise_type=noise_type,
        noise_param=noise_param
    )
    dataloader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        pin_memory=True if torch.cuda.is_available() else False,
        drop_last=False
    )
    base_names = [os.path.splitext(os.path.basename(f[0]))[0] for f in dataset.raw_rgb_pairs]
    return dataloader, base_names

# -------------------------- 图像增强 --------------------------
def enhance_raw_batch(
    input_raw: torch.Tensor, 
    model: RawFormer, 
    base_name: str,
    save_dir: str = ENHANCE_DIR,
    prefix: str = ENHANCED_PREFIX
) -> Tuple[Optional[np.ndarray], Optional[str], Optional[torch.Tensor]]:
    """增强RAW图像"""
    try:
        with torch.no_grad():
            pred_rgb_tensor = model(input_raw, cam_id=0)
            pred_rgb_tensor = torch.clamp(pred_rgb_tensor, 0, 1)
        
        pred_rgb_np = pred_rgb_tensor.cpu().numpy()
        pred_rgb_np = pred_rgb_np.squeeze()
        pred_rgb_np = pred_rgb_np.transpose((1, 2, 0))
        pred_rgb_np = (pred_rgb_np * 255).astype(np.uint8)
        
        enhanced_path = os.path.join(save_dir, f"{prefix}{base_name}.bmp")
        imageio.imwrite(enhanced_path, pred_rgb_np)
        
        print(f"增强图已保存：{os.path.basename(enhanced_path)}")
        return pred_rgb_np, enhanced_path, pred_rgb_tensor
    except Exception as e:
        print(f"错误：增强图像 {base_name} 失败 - {str(e)}")
        return None, None, None

# -------------------------- 指标计算 --------------------------
def calculate_psnr_train_style(gt_rgb_tensor: torch.Tensor, pred_rgb_tensor: torch.Tensor) -> float:
    """计算PSNR"""
    assert gt_rgb_tensor.shape == pred_rgb_tensor.shape, f"维度不匹配：gt={gt_rgb_tensor.shape}, pred={pred_rgb_tensor.shape}"
    assert gt_rgb_tensor.shape[0] == 1, f"batch维度错误：{gt_rgb_tensor.shape[0]}"
    assert gt_rgb_tensor.shape[1] == 3, f"通道维度错误：{gt_rgb_tensor.shape[1]}"
    
    gt_np = gt_rgb_tensor.squeeze(0).cpu().numpy().transpose(1, 2, 0)
    pred_np = pred_rgb_tensor.squeeze(0).detach().cpu().numpy().transpose(1, 2, 0)
    pred_np = np.clip(pred_np, 0.0, 1.0)
    
    gt_uint8 = (gt_np * 255).astype(np.uint8)
    pred_uint8 = (pred_np * 255).astype(np.uint8)
    
    psnr_value = psnr(gt_uint8, pred_uint8, data_range=255)
    return psnr_value

def calculate_ssim(gt_uint8: np.ndarray, pred_uint8: np.ndarray) -> float:
    """计算SSIM"""
    return ssim(
        gt_uint8, pred_uint8,
        data_range=255,
        channel_axis=2,
        gaussian_weights=True,
        sigma=1.5
    )

def calculate_lpips(gt_uint8: np.ndarray, pred_uint8: np.ndarray, lpips_model) -> float:
    """计算LPIPS"""
    def preprocess(img: np.ndarray) -> torch.Tensor:
        img_norm = (img / 255.0) * 2 - 1
        return torch.from_numpy(img_norm.transpose((2, 0, 1))).float().unsqueeze(0).to(DEVICE)
    
    gt_tensor = preprocess(gt_uint8)
    pred_tensor = preprocess(pred_uint8)
    
    with torch.no_grad():
        dist = lpips_model(gt_tensor, pred_tensor).item()
    return dist

def calculate_white_balance_error(gt_uint8: np.ndarray, pred_uint8: np.ndarray) -> float:
    """计算白平衡误差"""
    gt = gt_uint8.astype(np.float32) / 255.0
    pred = pred_uint8.astype(np.float32) / 255.0
    
    gt_r, gt_g, gt_b = np.mean(gt[..., 0]), np.mean(gt[..., 1]), np.mean(gt[..., 2])
    pred_r, pred_g, pred_b = np.mean(pred[..., 0]), np.mean(pred[..., 1]), np.mean(pred[..., 2])
    
    gt_g = max(gt_g, 1e-6)
    pred_g = max(pred_g, 1e-6)
    
    return abs((pred_r/pred_g) - (gt_r/gt_g)) + abs((pred_b/pred_g) - (gt_b/gt_g))

# -------------------------- 批量处理函数 --------------------------
def process_dataset(
    model: RawFormer,
    lpips_model,
    add_noise: bool = False,
    noise_type: str = "gaussian",
    noise_param: float = 0.01,
    save_dir: str = ENHANCE_DIR,
    prefix: str = ENHANCED_PREFIX
) -> Tuple[List[dict], float]:
    """处理数据集并返回结果和平均PSNR"""
    try:
        eval_dataloader, base_names = create_eval_dataset(
            add_noise=add_noise,
            noise_type=noise_type,
            noise_param=noise_param
        )
    except (ValueError, FileNotFoundError) as e:
        print(f"错误：{e}")
        return [], 0.0
    
    results = []
    total_samples = len(eval_dataloader)
    print(f"\n开始处理 {total_samples} 个样本（噪声类型：{noise_type}, 参数：{noise_param}）...")
    print("="*60)
    
    for idx, (input_raw, gt_rgb, gt_uint8, base_name) in enumerate(eval_dataloader, 1):
        print(f"\n[{idx}/{total_samples}] 处理样本：{base_name}")
        
        try:
            input_raw = input_raw.to(DEVICE)
            gt_rgb = gt_rgb.to(DEVICE)
            gt_uint8 = gt_uint8.squeeze().numpy()
            
            enhanced_img, enhanced_path, pred_rgb_tensor = enhance_raw_batch(
                input_raw, model, base_name, save_dir, prefix
            )
            if enhanced_img is None or pred_rgb_tensor is None:
                continue
            
            if gt_uint8.shape != enhanced_img.shape:
                print(f"警告：尺寸不匹配（GT: {gt_uint8.shape[:2]}, 增强图: {enhanced_img.shape[:2]}），跳过指标计算")
                continue
            
            psnr_val = calculate_psnr_train_style(gt_rgb, pred_rgb_tensor)
            ssim_val = calculate_ssim(gt_uint8, enhanced_img)
            lpips_val = calculate_lpips(gt_uint8, enhanced_img, lpips_model)
            wbe_val = calculate_white_balance_error(gt_uint8, enhanced_img)
            
            results.append({
                "样本名称": base_name,
                "噪声类型": noise_type if add_noise else "clean",
                "噪声参数": noise_param if add_noise else 0,
                "PSNR (dB)": round(psnr_val, 4),
                "SSIM": round(ssim_val, 4),
                "LPIPS": round(lpips_val, 4),
                "白平衡误差": round(wbe_val, 4),
                "增强图路径": enhanced_path
            })
            
            print(f"  PSNR: {psnr_val:.4f} dB | SSIM: {ssim_val:.4f} | LPIPS: {lpips_val:.4f} | WBE: {wbe_val:.4f}")
        
        except Exception as e:
            print(f"错误：处理样本 {base_name} 失败 - {str(e)}")
            continue
    
    if results:
        avg_psnr = np.mean([r["PSNR (dB)"] for r in results])
        print(f"\n平均PSNR: {avg_psnr:.4f} dB")
        return results, avg_psnr
    else:
        return [], 0.0

# -------------------------- 主流程 --------------------------
def main():
    print("="*60)
    print(f"开始噪声鲁棒性测试（RAW处理逻辑对齐Web版本）")
    print(f"RAW文件目录：{RAW_DIR}")
    print(f"GT文件目录：{RGB_DIR}")
    print("="*60)

    # 加载模型
    print("\n加载增强模型...")
    enhance_model = load_enhance_model()
    
    # 初始化LPIPS
    print(f"\n初始化LPIPS模型（{LPIPS_MODEL}）...")
    try:
        lpips_model = lpips.LPIPS(net=LPIPS_MODEL).to(DEVICE)
        lpips_model.eval()
    except Exception as e:
        print(f"错误：初始化LPIPS失败 - {str(e)}")
        return
    
    # 1. 处理干净数据（基准）
    print("\n" + "="*60)
    print("处理干净数据（基准测试）")
    clean_results, clean_avg_psnr = process_dataset(
        enhance_model,
        lpips_model,
        add_noise=False,
        save_dir=ENHANCE_DIR,
        prefix=ENHANCED_PREFIX
    )
    
    # 保存干净数据的评估结果
    if clean_results:
        df_clean = pd.DataFrame(clean_results)
        avg_row = {
            "样本名称": "平均值",
            "噪声类型": "clean",
            "噪声参数": 0,
            "PSNR (dB)": round(clean_avg_psnr, 4),
            "SSIM": round(np.mean([r["SSIM"] for r in clean_results]), 4),
            "LPIPS": round(np.mean([r["LPIPS"] for r in clean_results]), 4),
            "白平衡误差": round(np.mean([r["白平衡误差"] for r in clean_results]), 4),
            "增强图路径": ""
        }
        df_clean_with_avg = pd.concat([df_clean, pd.DataFrame([avg_row])], ignore_index=True)
        df_clean_with_avg.to_excel(OUTPUT_EXCEL, index=False, engine="openpyxl")
    
    # 2. 处理带噪声的数据
    noise_test_results = []
    all_results = clean_results.copy()
    
    for noise_type in NOISE_TYPES:
        print("\n" + "="*60)
        print(f"测试 {noise_type} 噪声")
        print("="*60)
        
        # 获取对应噪声类型的参数列表
        if noise_type == "gaussian":
            params = GAUSSIAN_NOISE_STD
            param_name = "标准差"
        elif noise_type == "poisson":
            params = POISSON_NOISE_SCALE
            param_name = "缩放因子"
        else:
            continue
        
        for param in params:
            noisy_results, noisy_avg_psnr = process_dataset(
                enhance_model,
                lpips_model,
                add_noise=True,
                noise_type=noise_type,
                noise_param=param,
                save_dir=NOISY_ENHANCE_DIR,
                prefix=f"{NOISY_ENHANCED_PREFIX}{noise_type}_{param}_"
            )
            
            all_results.extend(noisy_results)
            
            # 计算PSNR下降比例
            psnr_drop = (clean_avg_psnr - noisy_avg_psnr) / clean_avg_psnr * 100
            success = psnr_drop < 10  # 成功标准：下降<10%
            
            noise_test_results.append({
                "噪声类型": noise_type,
                f"噪声{param_name}": param,
                "干净数据平均PSNR (dB)": round(clean_avg_psnr, 4),
                "噪声数据平均PSNR (dB)": round(noisy_avg_psnr, 4),
                "PSNR下降值 (dB)": round(clean_avg_psnr - noisy_avg_psnr, 4),
                "PSNR下降比例 (%)": round(psnr_drop, 2),
                "是否满足成功标准(<10%)": "是" if success else "否"
            })
            
            print(f"\n{noise_type}噪声（{param_name}={param}）测试结果：")
            print(f"  干净数据平均PSNR: {clean_avg_psnr:.4f} dB")
            print(f"  噪声数据平均PSNR: {noisy_avg_psnr:.4f} dB")
            print(f"  PSNR下降比例: {psnr_drop:.2f}%")
            print(f"  是否满足成功标准: {'是' if success else '否'}")
    
    # 保存噪声测试汇总结果
    if noise_test_results:
        df_noise = pd.DataFrame(noise_test_results)
        df_noise.to_excel(NOISE_TEST_EXCEL, index=False, engine="openpyxl")
        
        # 保存所有结果的汇总
        df_all = pd.DataFrame(all_results)
        df_all.to_excel(os.path.join(BASE_DIR, "all_results.xlsx"), index=False, engine="openpyxl")
        
        print("\n" + "="*60)
        print("噪声鲁棒性测试完成！")
        print(f"📊 基准评估结果：{OUTPUT_EXCEL}")
        print(f"📊 噪声测试汇总：{NOISE_TEST_EXCEL}")
        print(f"📊 所有结果汇总：{os.path.join(BASE_DIR, 'all_results.xlsx')}")
        
        # 打印最终统计
        print("\n【噪声性能测试摘要】")
        print(f"干净数据平均PSNR: {clean_avg_psnr:.4f} dB")
        print("\n各噪声条件下的PSNR下降情况：")
        for result in noise_test_results:
            print(f"- {result['噪声类型']}({result[param_name]}={result[f'噪声{param_name}']}): "
                  f"{result['PSNR下降比例 (%)']}% (达标: {result['是否满足成功标准(<10%)']})")
        
        # 统计达标率
        success_count = sum(1 for r in noise_test_results if r["是否满足成功标准(<10%)"] == "是")
        total_tests = len(noise_test_results)
        print(f"\n总体达标率: {success_count}/{total_tests} ({success_count/total_tests*100:.1f}%)")
        
        # 检查是否所有测试都达标
        if success_count == total_tests:
            print("✅ 所有噪声测试均满足成功标准（PSNR下降<10%）")
        else:
            print("❌ 部分噪声测试未满足成功标准（PSNR下降≥10%）")
    else:
        print("\n❌ 没有成功处理任何噪声测试样本！")

if __name__ == "__main__":
    # 依赖检查
    try:
        import pandas
        import openpyxl
        import skimage
        import lpips
        import imageio
    except ImportError as e:
        missing_pkg = str(e).split("'")[1]
        print(f"缺少依赖包：{missing_pkg}")
        print("请执行以下命令安装：")
        print("pip install pandas openpyxl scikit-image lpips torch torchvision opencv-python numpy imageio")
        exit(1)
    
    main()

干净增强图保存到：eval_dataset/test/ENHANCE
噪声增强图保存到：eval_dataset/test/NOISY_ENHANCE
基础评估结果保存到：eval_dataset/test/enhanced_image_evaluation.xlsx
噪声测试结果保存到：eval_dataset/test/noise_performance_test.xlsx
开始噪声鲁棒性测试（RAW处理逻辑对齐Web版本）
RAW文件目录：eval_dataset/test/RAW
GT文件目录：eval_dataset/test/RGB

加载增强模型...
模型已加载到 cuda，权重路径：result/Freetech/weights/model_best.pth

初始化LPIPS模型（alex）...
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: /environment/miniconda3/envs/eval_env/lib/python3.12/site-packages/lpips/weights/v0.1/alex.pth

处理干净数据（基准测试）
找到 4 组有效配对文件

开始处理 4 个样本（噪声类型：gaussian, 参数：0.01）...

[1/4] 处理样本：['20251208-005734']
增强图已保存：enhanced_['20251208-005734'].bmp
  PSNR: 21.9845 dB | SSIM: 0.2845 | LPIPS: 0.5051 | WBE: 0.0638

[2/4] 处理样本：['20251208-005755']
增强图已保存：enhanced_['20251208-005755'].bmp
  PSNR: 24.0431 dB | SSIM: 0.3989 | LPIPS: 0.3492 | WBE: 12.4210

[3/4] 处理样本：['20251208-005828']
增强图已保存：enhanced_['20251208-005828'].bmp
  PSNR: 29.5703 dB | SSIM: 0.8087 | LP

KeyboardInterrupt: 